# B05 · Sesión 4 — Sistemas híbridos reglas/datos y guardarraíles

**Objetivo (RA5-b):** combinar reglas con aprendizaje automático. Dos enfoques: **deducir reglas de los datos** (FIGS) e **integrar reglas propias con ML** (Human-Learn). Cierre: reglas como **guardarraíl** de un LLM.

> Práctica guiada de la Sesión 4 de los [apuntes](../apuntes.md).

In [ ]:
%pip install imodels scikit-learn human-learn

## 1. Deducir reglas de los datos con FIGS

`FIGSClassifier` genera reglas `SI...ENTONCES` legibles a partir de un dataset. Interpretable, no caja negra.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from imodels import FIGSClassifier

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

clf = FIGSClassifier(max_rules=8)
clf.fit(X_tr, y_tr)
print("Accuracy FIGS:", round(clf.score(X_te, y_te), 3))
print(clf)

## 2. Reglas propias + ML con Human-Learn

`FunctionClassifier` envuelve una función de decisión tuya como si fuera un clasificador de scikit-learn.

In [ ]:
import numpy as np
try:
    from human_learn import FunctionClassifier
except ImportError:
    from hulearn.classification import FunctionClassifier

def regla_experto(X):
    # regla simple de experto: tumor grande (mean area, col 3) -> maligno
    return np.where(X[:, 3] > 800, 0, 1)

clf = FunctionClassifier(regla_experto)
clf.fit(X_tr, y_tr)
print("Accuracy regla de experto:", round(clf.score(X_te, y_te), 3))

## 3. Reglas como guardarraíl de un LLM

Un patrón muy actual: las reglas validan y acotan la salida de un modelo generativo antes de ejecutarla.

In [ ]:
import re

def validar_salida_llm(texto):
    """Aplica reglas de negocio a la salida de un LLM y devuelve (ok, motivo)."""
    # Regla 1: no se permiten importes > 1000 EUR
    importes = [float(x) for x in re.findall(r"(\d+(?:\.\d+)?)\s*EUR", texto)]
    if any(i > 1000 for i in importes):
        return False, "importe fuera de rango"
    # Regla 2: nunca transferir a cuentas no verificadas
    if "cuenta no verificada" in texto.lower():
        return False, "destino no permitido"
    return True, "ok"

for t in ["Transferir 500 EUR a la cuenta A", "Transferir 5000 EUR", "Pagar a cuenta no verificada"]:
    print(t, "->", validar_salida_llm(t))

## Actividad

1. Genera reglas con FIGS sobre *Iris* y compáralas con 2 reglas de experto que definas.
2. Añade una regla de guardarraíl extra a `validar_salida_llm` (p. ej. bloquear divisas no permitidas).

**Para casa:** ¿por qué un sistema neuro-simbólico reduce alucinaciones y mejora la explicabilidad?